In [ ]:
from pathlib import Path

p = Path("/content/drive/MyDrive/BrainAI_Project/backend_export/main.py")
print("Backend exists:", p.exists())
print("Path:", p)

Backend exists: False
Path: /content/drive/MyDrive/BrainAI_Project/backend_export/main.py


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from pathlib import Path

root = Path("/content/drive/MyDrive/BrainAI_Project")
print("Project folder exists:", root.exists())

if root.exists():
    for p in root.rglob("main.py"):
        print(p)

Project folder exists: True
/content/drive/MyDrive/BrainAI_Project/backend_export/main.py


In [ ]:
from pathlib import Path

p = Path("/content/drive/MyDrive/BrainAI_Project/backend_export/main.py")
text = p.read_text(encoding="utf-8")

print("FastAPI app found:", "FastAPI(" in text)
print("Existing RAG references:", "rag_router" in text or "/rag" in text)
print("\n--- First 35 lines ---\n")
print("\n".join(text.splitlines()[:35]))

FastAPI app found: True
Existing RAG references: False

--- First 35 lines ---

from pathlib import Path
import json
import base64
import uuid
import tempfile
import zipfile

import cv2
import nibabel as nib
import numpy as np
import tensorflow as tf

from fastapi import FastAPI, File, HTTPException, UploadFile
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse
from starlette.background import BackgroundTask

BASE_DIR = Path(__file__).resolve().parent
MODEL_PATH = BASE_DIR / "model" / "brain_tumor_unet_best.keras"
CONFIG_PATH = BASE_DIR / "model" / "model_config.json"

if not MODEL_PATH.exists():
    raise RuntimeError(f"Model file not found: {MODEL_PATH}")

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    CONFIG = json.load(f)

IMG_SIZE = int(CONFIG["input_size"][0])
MODALITIES = ["flair", "t1", "t1ce", "t2"]
THRESHOLD = float(CONFIG["prediction_threshold"])

model = tf.keras.models.load_model(MODEL_PATH, compile=False)

app = FastAPI(

In [ ]:
from pathlib import Path

root = Path("/content/drive/MyDrive/BrainAI_Project")
for p in root.rglob("rag.py"):
    print(p)

In [ ]:
from pathlib import Path

backend = Path("/content/drive/MyDrive/BrainAI_Project/backend_export")
backend.mkdir(parents=True, exist_ok=True)

rag_code = r'''
from pathlib import Path
import re

from fastapi import APIRouter
from pydantic import BaseModel, Field

router = APIRouter(prefix="/rag", tags=["RAG Assistant"])

BASE_DIR = Path(__file__).resolve().parent
KNOWLEDGE_DIR = BASE_DIR / "knowledge_base"


class QuestionRequest(BaseModel):
    question: str = Field(min_length=3, max_length=1000)


def load_documents():
    KNOWLEDGE_DIR.mkdir(parents=True, exist_ok=True)
    documents = []

    for path in KNOWLEDGE_DIR.rglob("*"):
        if path.suffix.lower() not in {".txt", ".md"}:
            continue
        try:
            content = path.read_text(encoding="utf-8").strip()
        except (OSError, UnicodeError):
            continue

        if content:
            documents.append({
                "source": path.name,
                "content": content
            })

    return documents


def tokenize(text):
    return set(re.findall(r"[a-zA-Z0-9]+", text.lower()))


def retrieve(question, documents, limit=3):
    question_words = tokenize(question)
    matches = []

    if not question_words:
        return matches

    for document in documents:
        paragraphs = [
            p.strip()
            for p in re.split(r"\n\s*\n", document["content"])
            if p.strip()
        ]

        for paragraph in paragraphs:
            passage_words = tokenize(paragraph)
            overlap = question_words & passage_words

            if overlap:
                score = len(overlap) / len(question_words)
                matches.append({
                    "source": document["source"],
                    "passage": paragraph[:1800],
                    "score": round(score, 3)
                })

    matches.sort(key=lambda item: item["score"], reverse=True)
    return matches[:limit]


@router.get("/health")
def rag_health():
    documents = load_documents()
    return {
        "status": "ready",
        "documents_loaded": len(documents)
    }


@router.post("/ask")
def ask_question(request: QuestionRequest):
    results = retrieve(request.question, load_documents())

    if not results:
        return {
            "answer": "I couldn't find relevant information in the current knowledge base.",
            "sources": []
        }

    evidence = "\n\n".join(
        f"[{i + 1}] {item['passage']}"
        for i, item in enumerate(results)
    )

    return {
        "answer": (
            "Relevant passages from the research collection are shown below. "
            "This version retrieves evidence rather than generating a new "
            "scientific answer.\n\n" + evidence
        ),
        "sources": [
            {"name": item["source"], "relevance": item["score"]}
            for item in results
        ]
    }
'''

rag_path = backend / "rag.py"
rag_path.write_text(rag_code.strip() + "\n", encoding="utf-8")
print("Created:", rag_path)

Created: /content/drive/MyDrive/BrainAI_Project/backend_export/rag.py


In [ ]:
from pathlib import Path

kb = Path(
    "/content/drive/MyDrive/BrainAI_Project/"
    "backend_export/knowledge_base"
)
kb.mkdir(parents=True, exist_ok=True)

content = """# Brain Tumors: Educational Reference

## Overview
A brain tumor is an abnormal growth of cells in or around the brain.
Tumors may be benign or malignant. MRI findings alone do not establish
a definitive diagnosis.

## MRI Modalities
- T1-weighted MRI helps show anatomical structures.
- T1-contrast-enhanced (T1ce) MRI can reveal regions that enhance after contrast.
- T2-weighted MRI highlights differences in tissue water content.
- FLAIR suppresses much of the cerebrospinal-fluid signal and helps
  visualize abnormalities near fluid spaces.

## MRI Segmentation
Segmentation identifies regions of interest within medical images at
the voxel level. A machine-learning model may produce a predicted mask,
but reliability depends on training data, image quality, scanner
differences, and validation.

## Important Limitations
AI-generated segmentation is a research aid, not a clinical diagnosis.
Predictions should be reviewed by qualified medical professionals.
Model confidence is not the same as diagnostic certainty.

## Evaluation
Segmentation models may be evaluated using Dice similarity coefficient,
intersection over union (IoU), sensitivity, and specificity. Evaluation
should use held-out data and avoid patient-level data leakage.
"""

doc = kb / "brain_tumor_info.md"
doc.write_text(content, encoding="utf-8")

print("Created:", doc)
print("Size:", doc.stat().st_size, "bytes")

Created: /content/drive/MyDrive/BrainAI_Project/backend_export/knowledge_base/brain_tumor_info.md
Size: 1247 bytes


In [ ]:
from pathlib import Path

main_path = Path(
    "/content/drive/MyDrive/BrainAI_Project/"
    "backend_export/main.py"
)

source = main_path.read_text(encoding="utf-8")

if "rag_router" in source:
    print("RAG router already appears in main.py; no changes made.")
else:
    import_line = "from rag import router as rag_router"
    include_line = "app.include_router(rag_router)"

    # Add the import after the existing imports.
    marker = "from starlette.background import BackgroundTask"
    if marker not in source:
        raise RuntimeError("Could not find the expected import marker.")

    source = source.replace(
        marker,
        marker + "\n" + import_line,
        1
    )

    # Add the router after the FastAPI app is created.
    marker = "app = FastAPI("
    start = source.find(marker)
    if start == -1:
        raise RuntimeError("Could not find FastAPI app declaration.")

    close = source.find("\n)", start)
    if close == -1:
        raise RuntimeError(
            "Could not safely identify the end of the FastAPI declaration."
        )

    close += 2
    source = (
        source[:close]
        + "\n\n"
        + include_line
        + source[close:]
    )

    # Save a backup before updating the file.
    backup = main_path.with_suffix(".py.before_rag")
    backup.write_text(
        main_path.read_text(encoding="utf-8"),
        encoding="utf-8"
    )
    main_path.write_text(source, encoding="utf-8")

    print("RAG router added.")
    print("Backup saved to:", backup)

RAG router added.
Backup saved to: /content/drive/MyDrive/BrainAI_Project/backend_export/main.py.before_rag


In [ ]:
from pathlib import Path
import ast

backend = Path(
    "/content/drive/MyDrive/BrainAI_Project/backend_export"
)

main_path = backend / "main.py"
rag_path = backend / "rag.py"
knowledge_path = backend / "knowledge_base" / "brain_tumor_info.md"

# Check syntax without executing or restarting the server.
ast.parse(main_path.read_text(encoding="utf-8"))
ast.parse(rag_path.read_text(encoding="utf-8"))

source = main_path.read_text(encoding="utf-8")

print("main.py syntax: OK")
print("rag.py syntax: OK")
print("RAG import connected:", "from rag import router as rag_router" in source)
print("Router registered:", "app.include_router(rag_router)" in source)
print("Knowledge document exists:", knowledge_path.exists())

main.py syntax: OK
rag.py syntax: OK
RAG import connected: True
Router registered: True
Knowledge document exists: True


In [ ]:
import requests

for url in [
    "http://127.0.0.1:8000/openapi.json",
    "http://127.0.0.1:8000/health",
]:
    try:
        r = requests.get(url, timeout=5)
        print(url, "->", r.status_code)
        if url.endswith("/openapi.json") and r.ok:
            paths = r.json().get("paths", {})
            print("RAG routes:", [p for p in paths if p.startswith("/rag")])
    except Exception as e:
        print(url, "->", str(e))

http://127.0.0.1:8000/openapi.json -> HTTPConnectionPool(host='127.0.0.1', port=8000): Max retries exceeded with url: /openapi.json (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7cc8f5cfe510>: Failed to establish a new connection: [Errno 111] Connection refused'))
http://127.0.0.1:8000/health -> HTTPConnectionPool(host='127.0.0.1', port=8000): Max retries exceeded with url: /health (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7cc8f5d12210>: Failed to establish a new connection: [Errno 111] Connection refused'))


In [ ]:
%cd /content/drive/MyDrive/BrainAI_Project/backend_export
!uvicorn main:app --host 127.0.0.1 --port 8000

/content/drive/MyDrive/BrainAI_Project/backend_export
I0000 00:00:1791751657.677692    9752 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1791751657.765787    9752 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1791751660.601940    9752 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
E0000 00:00:1791751663.041565    9752 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
INFO:     Started server process [9752]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)
INFO:     Shutting down
INFO:     Waiting for application s

In [ ]:
from pathlib import Path
import ast

backend = Path("/content/drive/MyDrive/BrainAI_Project/backend_export")
main_path = backend / "main.py"
rag_path = backend / "rag.py"

source = main_path.read_text(encoding="utf-8")
ast.parse(source)
ast.parse(rag_path.read_text(encoding="utf-8"))

print("RAG file exists:", rag_path.exists())
print("RAG import:", "from rag import router as rag_router" in source)
print("Router registration:", "app.include_router(rag_router)" in source)
print("App declaration and router lines:")
for i, line in enumerate(source.splitlines(), 1):
    if "FastAPI(" in line or "rag_router" in line:
        print(i, line)

RAG file exists: True
RAG import: True
Router registration: True
App declaration and router lines:
17 from rag import router as rag_router
35 app = FastAPI(
44 app.include_router(rag_router)


In [ ]:
%cd /content/drive/MyDrive/BrainAI_Project/backend_export
!uvicorn main:app --host 127.0.0.1 --port 8000

/content/drive/MyDrive/BrainAI_Project/backend_export
I0000 00:00:1791751883.515771   10679 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1791751883.631510   10679 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1791751887.310846   10679 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
E0000 00:00:1791751889.771637   10679 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
INFO:     Started server process [10679]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)
INFO:     Shutting down
INFO:     Finished server process 